# 作业 01：一个预测函数，接收多个输入张量

上周的预测函数只有一个输入 x。这次同一个样本有两个输入 x1、x2，它们共同决定一个目标 y：

$$\hat y = w_1x_1 + w_2x_2 + b.$$

重点：调用一次预测函数时，需要同时传入 x1 和 x2。它们不是两次独立预测，也不是两个不同样本；例如同一个物体的两个测量值，都可能影响预测结果。

下面还会把多个样本分别放进 x1、x2 和 y。`x1[i]`、`x2[i]`、`y[i]` 属于同一个样本。请区分“一个样本有两个输入”和“共有多少个样本”。

本题仅需 CPU 和 PyTorch。训练步骤沿用上周：预测、损失、清空梯度、反向传播、更新参数。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b

# 同一个样本的两个输入，预测时必须一起使用
x1_demo = torch.tensor(1.0)
x2_demo = torch.tensor(2.0)
y_demo = torch.tensor(-3.0)
w1_demo = torch.tensor(1.0)
w2_demo = torch.tensor(1.0)
b_demo = torch.tensor(0.0)
y_pred_demo = predict(x1_demo, x2_demo, w1_demo, w2_demo, b_demo)
loss_demo = torch.abs(y_demo - y_pred_demo)
for name, value in [('x1', x1_demo), ('x2', x2_demo), ('y', y_demo),
                    ('y_pred', y_pred_demo), ('loss', loss_demo)]:
    print(name, 'shape =', tuple(value.shape), 'value =', value)



x1 shape = () value = tensor(1.)
x2 shape = () value = tensor(2.)
y shape = () value = tensor(-3.)
y_pred shape = () value = tensor(3.)
loss shape = () value = tensor(6.)


## 同一个样本，也可以只用一个 x 张量

上面的 x1_demo 和 x2_demo 分别是两个标量张量，shape 都为 `()`。现在直接用 `torch.tensor([1.0, 2.0])` 创建一个张量 x，shape 就是 `(2,)`：**一个样本，包含两个输入值**。这里的 2 表示两个特征，不是两个样本。

预测公式和参数都不变，只改变函数接收输入的方式：原来传入 x1、x2；现在传入 x，再用 x[0]、x[1] 取出两个值。此处只比较单个样本，不进行训练；多样本的训练代码放在作业 02。


In [2]:
# 直接创建一个包含两个输入值的张量，表示同一个样本
x = torch.tensor([1.0, 2.0])

def predict_packed(x, w1, w2, b):
    return w1 * x[0] + w2 * x[1] + b

y_pred_packed = predict_packed(x, w1_demo, w2_demo, b_demo)
loss_packed = torch.abs(y_demo - y_pred_packed)

for name, value in [('x1_demo', x1_demo), ('x2_demo', x2_demo),
                    ('x', x), ('x[0]', x[0]), ('x[1]', x[1]),
                    ('y', y_demo), ('y_pred_packed', y_pred_packed),
                    ('loss_packed', loss_packed)]:
    print(name, 'shape =', tuple(value.shape), 'value =', value)

print('两个输入张量的预测：', y_pred_demo)
print('一个打包张量的预测：', y_pred_packed)
assert x.shape == (2,)
assert torch.allclose(y_pred_demo, y_pred_packed)
assert torch.allclose(loss_demo, loss_packed)


x1_demo shape = () value = tensor(1.)
x2_demo shape = () value = tensor(2.)
x shape = (2,) value = tensor([1., 2.])
x[0] shape = () value = tensor(1.)
x[1] shape = () value = tensor(2.)
y shape = () value = tensor(-3.)
y_pred_packed shape = () value = tensor(3.)
loss_packed shape = () value = tensor(6.)
两个输入张量的预测： tensor(3.)
一个打包张量的预测： tensor(3.)


## 改变一个输入值，比较两种写法

先预测：只把第二个输入加 1，两种写法的预测是否仍然相同？第一个输入是否改变？

下面始终使用同一组参数，不进行训练。用 clone 复制 x 后再修改，以保留原始样本，方便比较。


In [3]:
x_changed = x.clone()
x_changed[1] = x_changed[1] + 1.0

y_pred_separate_changed = predict(
    x1_demo, x2_demo + 1.0, w1_demo, w2_demo, b_demo)
y_pred_packed_changed = predict_packed(
    x_changed, w1_demo, w2_demo, b_demo)

print('原始 x：', x, 'shape：', tuple(x.shape))
print('修改后 x：', x_changed, 'shape：', tuple(x_changed.shape))
print('修改前的预测：', y_pred_packed)
print('修改后，两个输入张量的预测：', y_pred_separate_changed)
print('修改后，一个打包张量的预测：', y_pred_packed_changed)
assert torch.allclose(y_pred_separate_changed, y_pred_packed_changed)


原始 x： tensor([1., 2.]) shape： (2,)
修改后 x： tensor([1., 3.]) shape： (2,)
修改前的预测： tensor(3.)
修改后，两个输入张量的预测： tensor(4.)
修改后，一个打包张量的预测： tensor(4.)


## 我的观察

1. 两个标量张量的 shape 分别是什么？打包后的 x 的 shape 是什么？x[0] 和 x[1] 的 shape 又是什么？（20分）
2. x 的 shape 为 `(2,)`，在本例中代表两个样本，还是一个样本的两个输入值？（10分）
3. 比较 predict 与 predict_packed：数据的组织和传入方式如何改变？预测公式、参数、y 和 loss 是否改变？（20分）

1. `x1_demo` 和 `x2_demo` 都是标量张量，shape 分别为 `()` 和 `()`。打包后的 `x` 的 shape 为 `(2,)`；索引取出的 `x[0]`、`x[1]` 仍是标量，shape 都为 `()`。`y`、预测值和绝对值损失也都是标量，shape 为 `()`。

2. 本例中 `(2,)` 表示**一个样本的两个输入值（两个特征）**，不是两个样本。这两个数分别对应同一个样本的 `x1` 和 `x2`。

3. `predict` 分别接收 `x1`、`x2` 两个输入张量；`predict_packed` 接收一个包含两个值的张量 `x`，再通过 `x[0]`、`x[1]` 取出特征。两者只改变数据的组织和传入方式，预测公式仍是 $\hat y=w_1x_1+w_2x_2+b$，参数都为 `w1=1、w2=1、b=0`，真实目标都为 `y=-3`。原始输入为 `[1, 2]` 时，两种写法均得到预测 `3`，绝对值损失均为 `|-3-3|=6`。

   只将第二个输入从 `2` 改为 `3` 后，第一个输入仍为 `1`，两种写法的预测都变为 `4`。由于 `w2=1`，第二个输入增加 `1`，预测也增加 `1`；若继续使用同一个目标 `y=-3`，对应损失为 `|-3-4|=7`。使用 `clone()` 后修改的是副本，原始 `x` 仍为 `[1, 2]`。
